In [4]:
import os
import cv2
import torch
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

# Paths for original and manipulated videos
original_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences"
manipulated_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences"

# Collect video paths and labels
original_videos = [os.path.join(original_videos_dir, filename) for filename in os.listdir(original_videos_dir)]
manipulated_videos = [os.path.join(manipulated_videos_dir, filename) for filename in os.listdir(manipulated_videos_dir)]

original_labels = [0] * len(original_videos)  # 0 for original videos
manipulated_labels = [1] * len(manipulated_videos)  # 1 for manipulated videos

all_videos = original_videos + manipulated_videos
labels = original_labels + manipulated_labels


class DeepfakeDataset(Dataset):
    def __init__(self, videos, labels, processor, frame_count=5, transform=None):
        self.videos = videos
        self.labels = labels
        self.processor = processor
        self.frame_count = frame_count
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        video_path = self.videos[idx]
        label = self.labels[idx]

        
    # Extract frames from video
        cap = cv2.VideoCapture(video_path)
        frames = []
        for _ in range(self.frame_count):
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            frames.append(self.transform(frame))
        cap.release()
    
        # Handle empty frames
        if len(frames) == 0:
            # Add default blank frames of size [3, 224, 224]
            blank_frame = torch.zeros(3, 224, 224)  # RGB with height and width
            frames = [blank_frame] * self.frame_count
    # Pad frames if less than required
        while len(frames) < self.frame_count:
            frames.append(torch.zeros_like(frames[0]))
    
        # Stack frames into a tensor and aggregate
        frames_tensor = torch.stack(frames)
        aggregated_frame = frames_tensor.mean(dim=0)
    
        # Ensure the pixel values are within [0, 255]
        aggregated_frame = aggregated_frame * 255  # Scale to [0, 255]
        aggregated_frame = aggregated_frame.clamp(0, 255).byte()  # Convert to uint8
    
        # Process the aggregated frame using the processor
        inputs = self.processor(images=aggregated_frame, return_tensors="pt", do_rescale=False)
        pixel_values = inputs['pixel_values'].squeeze(0)
    
        return pixel_values, torch.tensor(label)



# Initialize Dataset and DataLoader
processor = AutoImageProcessor.from_pretrained("Wvolf/ViT_Deepfake_Detection")
train_videos, val_videos, train_labels, val_labels = train_test_split(
    all_videos, labels, test_size=0.2, random_state=42, stratify=labels
)
train_dataset = DeepfakeDataset(train_videos, train_labels, processor)
val_dataset = DeepfakeDataset(val_videos, val_labels, processor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


# Model Setup
model = AutoModelForImageClassification.from_pretrained("Wvolf/ViT_Deepfake_Detection")
model.config.num_labels = 2
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# Training Loop
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
device = model.device


def evaluate(model, val_loader, criterion):
    model.eval()
    val_loss = 0.0
    val_accuracy = 0.0
    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            outputs = model(pixel_values=pixel_values)
            loss = criterion(outputs.logits, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.logits, 1)
            val_accuracy += (predicted == labels).sum().item() / labels.size(0)
    return val_loss / len(val_loader), val_accuracy / len(val_loader)

for epoch in range(5):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for pixel_values, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        pixel_values, labels = pixel_values.to(device), labels.to(device)
         # Forward pass
        outputs = model(pixel_values=pixel_values)
        loss = criterion(outputs.logits, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update total loss
        total_loss += loss.item()

        # Calculate batch accuracy
        _, predicted = torch.max(outputs.logits, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    # Calculate average training loss and accuracy
    avg_train_loss = total_loss / len(train_loader)
    train_accuracy = total_correct / total_samples

    # Evaluate on validation data
    val_loss, val_accuracy = evaluate(model, val_loader, criterion)

    # Print training and validation metrics
    print(f"Epoch {epoch+1}: "
          f"Train Loss = {avg_train_loss:.4f}, Train Acc = {train_accuracy:.4f}, "
          f"Val Loss = {val_loss:.4f}, Val Acc = {val_accuracy:.4f}")

# Save the Model
model.save_pretrained("fine_tuned_deepfake_vit")
processor.save_pretrained("fine_tuned_deepfake_vit")

Epoch 1: 100%|██████████| 43/43 [06:32<00:00,  9.14s/it]


Epoch 1: Train Loss = 0.3565, Train Acc = 0.8944, Val Loss = 0.3384, Val Acc = 0.8953


Epoch 2: 100%|██████████| 43/43 [06:32<00:00,  9.12s/it]


Epoch 2: Train Loss = 0.3385, Train Acc = 0.8944, Val Loss = 0.3407, Val Acc = 0.8953


Epoch 3: 100%|██████████| 43/43 [06:32<00:00,  9.13s/it]


...

Epoch 4: 100%|██████████| 43/43 [06:32<00:00,  9.14s/it]


Epoch 4: Train Loss = 0.3353, Train Acc = 0.8944, Val Loss = 0.3379, Val Acc = 0.8953


Epoch 5: 100%|██████████| 43/43 [06:34<00:00,  9.17s/it]


Epoch 5: Train Loss = 0.3335, Train Acc = 0.8944, Val Loss = 0.3377, Val Acc = 0.8953


['fine_tuned_deepfake_vit/preprocessor_config.json']

In [11]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import shap
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, AutoImageProcessor, AutoModelForImageClassification
from torchvision import transforms
from torchvision.utils import save_image
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving images

# Create output directory for saving visualizations
output_dir = "shap_visualizations"
os.makedirs(output_dir, exist_ok=True)

# Load BLIP model for image captioning
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model.eval()

# Load the fine-tuned ViT model and processor
model = AutoModelForImageClassification.from_pretrained("fine_tuned_deepfake_vit")
processor = AutoImageProcessor.from_pretrained("fine_tuned_deepfake_vit")
model.eval()
model.cpu()  # SHAP works better on CPU

# Define transforms for video frame processing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Custom Dataset
class DeepfakeDataset(torch.utils.data.Dataset):
    def __init__(self, videos, labels, processor, frame_count=5, transform=None):
        self.videos = videos
        self.labels = labels
        self.processor = processor
        self.frame_count = frame_count
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        video_path = self.videos[idx]
        label = self.labels[idx]
        
        cap = cv2.VideoCapture(video_path)
        frames = []
        for _ in range(self.frame_count):
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            frames.append(self.transform(frame))
        cap.release()
        
        if len(frames) == 0:
            blank_frame = torch.zeros(3, 224, 224)
            frames = [blank_frame] * self.frame_count
        while len(frames) < self.frame_count:
            frames.append(torch.zeros_like(frames[0]))
        
        # Use the first frame to preserve channels
        frames_tensor = torch.stack(frames)
        aggregated_frame = frames_tensor[0]
        
        # Ensure pixel values are within [0, 255]
        aggregated_frame = aggregated_frame * 255
        aggregated_frame = aggregated_frame.clamp(0, 255).byte()
        
        # Process the aggregated frame
        inputs = self.processor(images=aggregated_frame, return_tensors="pt", do_rescale=False)
        pixel_values = inputs['pixel_values'].squeeze(0)
        
        return pixel_values, torch.tensor(label)

# Define prediction function for SHAP
def predict_for_shap(images):
    try:
        if isinstance(images, list):
            images = np.stack(images)  # Convert list to numpy array
        if isinstance(images, np.ndarray):
            images = torch.tensor(images, dtype=torch.float32)
        if images.dim() == 3:
            images = images.unsqueeze(0)
        if images.shape[-1] == 3:
            images = images.permute(0, 3, 1, 2)
        images = images.cpu()
        with torch.no_grad():
            outputs = model(pixel_values=images)
            return outputs.logits.numpy()
    except Exception as e:
        print(f"Prediction error: {str(e)}")
        return np.zeros((len(images), 2))

# Generate caption using BLIP
def generate_caption(image_tensor):
    try:
        # Ensure image_tensor is in [0, 1] and correct shape
        if isinstance(image_tensor, torch.Tensor):
            image_tensor = image_tensor.clamp(0, 1)
            if image_tensor.dim() == 3:
                if image_tensor.shape[0] == 1:  # Single channel, replicate to 3
                    image_tensor = image_tensor.repeat(3, 1, 1)
                elif image_tensor.shape[0] != 3:
                    raise ValueError(f"Unexpected channel dimension: {image_tensor.shape}")
                pil_img = Image.fromarray((image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
            elif image_tensor.dim() == 4 and image_tensor.shape[1] == 3:
                pil_img = Image.fromarray((image_tensor[0].permute(1, 2, 0).numpy() * 255).astype(np.uint8))
            else:
                raise ValueError(f"Unexpected tensor shape: {image_tensor.shape}")
        else:
            pil_img = Image.fromarray((image_tensor * 255).astype(np.uint8))
        
        inputs = blip_processor(pil_img, return_tensors="pt")
        with torch.no_grad():
            generated_ids = blip_model.generate(**inputs, max_length=30)
            caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        return caption
    except Exception as e:
        print(f"Caption generation error: {str(e)}")
        return "Unable to generate caption"

# Visualize SHAP values manually
def visualize_shap_manually(shap_values, images, index):
    try:
        if isinstance(shap_values, list):
            shap_image = shap_values[0][index]  # Take first class explanation
        else:
            shap_image = shap_values[index].values if hasattr(shap_values[index], 'values') else shap_values[index]
        
        # Ensure SHAP image is in the correct shape
        if shap_image.ndim == 4:  # (1, C, H, W) or (N, C, H, W)
            shap_image = shap_image[0]  # Take the first sample
        if shap_image.shape[0] == 3:  # CHW format
            abs_shap = np.abs(shap_image).sum(axis=0)  # Sum over channels
        else:  # HWC format
            abs_shap = np.abs(shap_image).sum(axis=-1)
        
        max_val = np.max(abs_shap)
        if max_val > 0:
            abs_shap = abs_shap / max_val
        heatmap = plt.cm.hot(abs_shap)[:, :, :3]  # Remove alpha channel
        
        # Get original image
        original = images[index].transpose(1, 2, 0) if images[index].shape[0] == 3 else images[index]
        # Ensure shapes match
        if heatmap.shape != original.shape:
            heatmap = cv2.resize(heatmap, (original.shape[1], original.shape[0]))
        
        # Blend the heatmap with the original image
        overlay = original * 0.5 + heatmap * 0.5
        return overlay.clip(0, 1)  # Ensure values are in [0, 1]
    except Exception as e:
        print(f"Visualization error for sample {index}: {str(e)}")
        return images[index].transpose(1, 2, 0) if images[index].shape[0] == 3 else images[index]

# Save SHAP visualizations
def save_shap_visualizations(shap_values, images, labels, output_dir):
    for i in range(len(images)):
        try:
            label_text = "Real" if labels[i] == 0 else "Manipulated"
            prediction = predict_for_shap([images[i]])[0]  # Pass as list to handle single image
            pred_label = "Real" if np.argmax(prediction) == 0 else "Manipulated"
            confidence = np.max(prediction) / np.sum(np.abs(prediction)) * 100 if np.sum(np.abs(prediction)) > 0 else 0.0
            caption = generate_caption(images[i])
            shap_overlay = visualize_shap_manually(shap_values, images, i)
            
            plt.figure(figsize=(14, 7))
            plt.subplot(1, 2, 1)
            plt.imshow(images[i].transpose(1, 2, 0) if images[i].shape[0] == 3 else images[i])
            plt.title(f"Original Frame\nTrue: {label_text}, Pred: {pred_label} ({confidence:.1f}%)\n{caption}")
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(shap_overlay)
            plt.title("SHAP Values")
            plt.axis('off')
            
            filename = os.path.join(output_dir, f"shap_sample_{i}_label_{label_text}.png")
            plt.savefig(filename, bbox_inches='tight', dpi=150)
            plt.close()
            
            print(f"Saved visualization for sample {i} to {filename}")
            print(f"  - True label: {label_text}")
            print(f"  - Predicted: {pred_label} (Confidence: {confidence:.1f}%)")
            print(f"  - Caption: {caption}")
        except Exception as e:
            print(f"Failed to process sample {i}: {str(e)}")

# Analyze video sequence
def analyze_video_sequence(video_path, processor, frame_count=5):
    try:
        video_name = os.path.basename(video_path).split('.')[0]
        video_output_dir = os.path.join(output_dir, f"video_{video_name}")
        os.makedirs(video_output_dir, exist_ok=True)
        
        cap = cv2.VideoCapture(video_path)
        frames = []
        for _ in range(frame_count):
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            frame_tensor = transform(frame)
            frames.append(frame_tensor)
        cap.release()
        
        if not frames:
            print(f"No frames extracted from {video_path}")
            return
        
        frames_tensor = torch.stack(frames)
        aggregated_frame = frames_tensor[0]  # Use first frame to preserve channels
        aggregated_frame = aggregated_frame.clamp(0, 1)
        
        with torch.no_grad():
            inputs = processor(images=aggregated_frame, return_tensors="pt", do_rescale=False)
            outputs = model(pixel_values=inputs['pixel_values'])
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=1).item()
        
        caption = generate_caption(aggregated_frame)
        shap_values = explainer(aggregated_frame.unsqueeze(0).numpy())
        shap_overlay = visualize_shap_manually(shap_values, [aggregated_frame.numpy()], 0)
        
        plt.figure(figsize=(14, 7))
        plt.subplot(1, 2, 1)
        plt.imshow(aggregated_frame.permute(1, 2, 0))
        pred_text = "Real" if prediction == 0 else "Manipulated"
        plt.title(f"Video: {video_name}\nPrediction: {pred_text}\n{caption}")
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.imshow(shap_overlay)
        plt.title("SHAP Values")
        plt.axis('off')
        
        filename = os.path.join(video_output_dir, f"analysis_video_{video_name}.png")
        plt.savefig(filename, bbox_inches='tight', dpi=150)
        plt.close()
        
        print(f"Saved video analysis to {filename}")
        print(f"  - Caption: {caption}")
        print(f"  - Prediction: {pred_text}")
    except Exception as e:
        print(f"Error processing video {video_path}: {str(e)}")

# Initialize dataset and collect samples
original_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences"
manipulated_videos_dir = "/kaggle/input/deep-fake-detection-dfd-entire-original-dataset/DFD_manipulated_sequences/DFD_manipulated_sequences"

original_videos = [os.path.join(original_videos_dir, filename) for filename in os.listdir(original_videos_dir)]
manipulated_videos = [os.path.join(manipulated_videos_dir, filename) for filename in os.listdir(manipulated_videos_dir)]
original_labels = [0] * len(original_videos)
manipulated_labels = [1] * len(manipulated_videos)
all_videos = original_videos + manipulated_videos
labels = original_labels + manipulated_labels

from sklearn.model_selection import train_test_split
_, val_videos, _, val_labels = train_test_split(all_videos, labels, test_size=0.2, random_state=42, stratify=labels)

val_dataset = DeepfakeDataset(val_videos, val_labels, processor, transform=transform)
num_samples = min(5, len(val_dataset))

sample_frames = []
sample_labels = []
for i in range(num_samples):
    img_tensor, label = val_dataset[i]
    img_tensor = img_tensor / 255.0  # Normalize to [0, 1]
    sample_frames.append(img_tensor.numpy())
    sample_labels.append(label.item())

sample_frames_tensor = torch.stack([torch.tensor(f) for f in sample_frames])
if sample_frames_tensor.shape[1] != 3:
    sample_frames_tensor = sample_frames_tensor.permute(0, 3, 1, 2)
sample_frames_tensor = sample_frames_tensor.clamp(0, 1)

# Create SHAP explainer
try:
    sample_image = sample_frames[0]
    if sample_image.shape[0] == 3:
        sample_image = sample_image.transpose(1, 2, 0)
    sample_image_uint8 = (sample_image * 255).astype(np.uint8)
    
    masker = shap.maskers.Image("blur(28,28)", sample_image_uint8.shape)
    explainer = shap.Explainer(
        predict_for_shap,
        masker,
        output_names=["Real", "Manipulated"],
        max_evals=100
    )
    shap_values = explainer(sample_frames_tensor)
    print("Successfully computed SHAP values with PartitionExplainer")
except Exception as e:
    print(f"Partition Explainer failed: {str(e)}")
    print("Trying GradientExplainer...")
    try:
        background = sample_frames_tensor[:1]
        explainer = shap.GradientExplainer(
            model,
            background,
            batch_size=1,
            local_smoothing=0.0
        )
        shap_values = explainer.shap_values(sample_frames_tensor)
        print("Successfully computed SHAP values with GradientExplainer")
    except Exception as e:
        print(f"Gradient Explainer failed: {str(e)}")
        raise RuntimeError("Both Partition and Gradient Explainers failed")

# Save visualizations
save_shap_visualizations(shap_values, sample_frames, sample_labels, output_dir)

# Analyze videos
for video_path in val_videos[:3]:
    analyze_video_sequence(video_path, processor)

print(f"Analysis complete. Results saved to {output_dir}")

`clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884


  0%|          | 0/98 [00:00<?, ?it/s]

PartitionExplainer explainer:  20%|██        | 1/5 [00:00<?, ?it/s]

  0%|          | 0/98 [00:00<?, ?it/s]

PartitionExplainer explainer:  60%|██████    | 3/5 [00:33<00:16,  8.28s/it]

...

PartitionExplainer explainer: 2it [00:14, 14.87s/it]               


Saved video analysis to shap_visualizations/video_02_15__meeting_serious__N864L40U/analysis_video_02_15__meeting_serious__N864L40U.png
  - Caption: three women sitting at a table talking
  - Prediction: Real


  0%|          | 0/98 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:15, 15.91s/it]               


Saved video analysis to shap_visualizations/video_27_02__walk_down_hall_angry__FV8M8O2C/analysis_video_27_02__walk_down_hall_angry__FV8M8O2C.png
  - Caption: a woman walking down a long hallway
  - Prediction: Real
Analysis complete. Results saved to shap_visualizations
